# Convert Arrow to CSV with chDB

Companion to the article *How to convert Arrow to CSV*. Run `./generate.sh` first to create `data/events.arrow`.

In [ ]:
import os
import chdb

os.chdir(os.path.join(os.getcwd(), "data"))

## 1. Convert Arrow -> CSV in one line
The schema is read from the Arrow file; no structure has to be supplied.

In [ ]:
chdb.query(
    "SELECT * FROM file('events.arrow') "
    "INTO OUTFILE 'events_chdb.csv' TRUNCATE FORMAT CSVWithNames"
)
with open("events_chdb.csv") as f:
    print("\n".join(f.read().splitlines()[:4]))

## 2. The nested Array column flattens to a quoted string in CSV (the gotcha)

In [ ]:
print(chdb.query("SELECT event_id, tags FROM file('events_chdb.csv') LIMIT 3", "CSV"), end="")

## 3. Transform during conversion, straight to a DataFrame

In [ ]:
df = chdb.query(
    "SELECT country, count() AS events, round(sum(amount),2) AS total "
    "FROM file('events.arrow') GROUP BY country ORDER BY total DESC",
    "DataFrame",
)
df